In [1]:
import pandas as pd
import json
import openpyxl

In [2]:
from pathlib import Path

path = Path(r"_site-traffic-volume-map\widgets\ForecastSidebar\data\segments.json")
if not path.exists():
  raise FileNotFoundError(f"JSON file not found: {path.resolve()}")

with path.open("r", encoding="utf-8") as f:
  segments = json.load(f)

print(f"Loaded JSON type: {type(segments)}")
if isinstance(segments, dict):
  print("Top-level keys:", list(segments.keys()))
elif isinstance(segments, list):
  print("List length:", len(segments))

# If it's a list of records, convert to a DataFrame for easier inspection
if isinstance(segments, list) and segments and isinstance(segments[0], dict):
  segments_df = pd.DataFrame(segments)
  print("Converted to DataFrame with shape:", segments_df.shape)
else:
  # If the useful data is nested under a key, try to find the first list of dicts
  for k, v in (segments.items() if isinstance(segments, dict) else []):
    if isinstance(v, list) and v and isinstance(v[0], dict):
      segments_df = pd.DataFrame(v)
      print(f"Found list under key '{k}' -> DataFrame shape:", segments_df.shape)
      break

segments_df['MP'] = segments_df['S'].str.split('_').str[1].astype(float)

segments_df = segments_df[['S','R','MP']].drop_duplicates().reset_index(drop=True)
segments_df.sort_values(by=['S'], inplace=True)
segments_df

Loaded JSON type: <class 'list'>
List length: 18883
Converted to DataFrame with shape: (18883, 5)


,S,R,MP
5040,0006_000.0,0006,0.0
5041,0006_000.7,0006,0.7
5042,0006_016.0,0006,16.0
5043,0006_046.0,0006,46.0
5044,0006_060.2,0006,60.2
...,...,...,...
4999,WFRC_8469,WFRC,8469.0
5000,WFRC_8470,WFRC,8470.0
5001,WFRC_8471,WFRC,8471.0
5002,WFRC_8472,WFRC,8472.0


In [3]:
# locate the workbook (searching cwd recursively so the exact path/escape issues don't matter)
candidates = list(Path.cwd().rglob("Truck Traffic on Utah Highways 2023.xlsx"))
if not candidates:
  candidates = list(Path.cwd().rglob("*Truck Traffic on Utah Highways*.xlsx"))
if not candidates:
  raise FileNotFoundError("Excel file not found under cwd. Adjust filename or path.")
excel_path = candidates[0]
print("Using workbook:", excel_path)

# read first sheet (no header enforcement so we can inspect and detect real header row)
truck_df_raw = pd.read_excel(excel_path, sheet_name=0)
print("Raw shape:", truck_df_raw.shape)
#display(truck_df_raw.head(10))
truck_df_raw.info()

# attempt to auto-detect header row: choose first row with >=50% non-null cells
header_row = None
max_check = min(10, len(truck_df_raw))
for i in range(max_check):
  if truck_df_raw.iloc[i].notna().sum() >= (len(truck_df_raw.columns) / 2):
    header_row = i
    break
if header_row is None:
  header_row = 0
print("Detected header row:", header_row)

# re-read with detected header (or use the raw if header_row == 0)
if header_row != 0:
  truck_df = pd.read_excel(excel_path, sheet_name=0, header=header_row)
else:
  truck_df = truck_df_raw.copy()

# basic cleanup: strip column names, drop fully-empty rows/cols, reset index
truck_df.columns = truck_df.columns.astype(str).str.strip()
truck_df = truck_df.dropna(axis=0, how="all").dropna(axis=1, how="all").reset_index(drop=True)

# coerce object columns to numeric when appropriate (if at least half the column can become numeric)
for col in truck_df.columns:
  if truck_df[col].dtype == object:
    coerced = pd.to_numeric(truck_df[col], errors="coerce")
    if coerced.notna().sum() >= (len(truck_df) / 2):
      truck_df[col] = coerced


truck_df['R'] = truck_df['ROUTE_ID'].astype(str).str[:4]

truck_df['FROM_MEASURE'] = truck_df['FROM_MEASURE'].round(1)
truck_df['TO_MEASURE'  ] = truck_df['TO_MEASURE'  ].round(1)

truck_df = truck_df[['R','FROM_MEASURE','TO_MEASURE','AADT2023','SUTRK2023','CUTRK2023']]

print("Cleaned sheet shape:", truck_df.shape)
display(truck_df)

Using workbook: d:\GitHub\FORECAST-Traffic-Volume\data\udot\Truck Traffic on Utah Highways 2023.xlsx
Raw shape: (1877, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1877 entries, 0 to 1876
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   STATION       1868 non-null   object 
 1   ROUTE_ID      1877 non-null   object 
 2   FROM_MEASURE  1877 non-null   float64
 3   TO_MEASURE    1877 non-null   float64
 4   DESC          1874 non-null   object 
 5   AADT2023      1877 non-null   int64  
 6   SUTRK2023     1876 non-null   float64
 7   CUTRK2023     1876 non-null   float64
dtypes: float64(4), int64(1), object(3)
memory usage: 117.4+ KB
Detected header row: 0
Cleaned sheet shape: (1877, 6)


,R,FROM_MEASURE,TO_MEASURE,AADT2023,SUTRK2023,CUTRK2023
0,0006,0.0,46.0,457,0.249610,0.232449
1,0006,46.0,77.6,409,0.175063,0.333819
2,0006,77.6,82.9,586,0.162481,0.264722
3,0006,82.9,83.9,2189,0.149898,0.195620
4,0006,83.9,87.7,4012,0.137314,0.126513
...,...,...,...,...,...,...
1872,0319,0.0,0.9,1508,0.094570,0.254773
1873,0491,0.0,0.5,6445,0.063942,0.330758
1874,0491,0.5,2.0,3812,0.063942,0.330758
1875,0491,2.0,17.1,3558,0.063942,0.330758


In [4]:
merged_df = (
    segments_df.merge(truck_df, on='R', how='inner')
    .query("MP >= FROM_MEASURE and MP < TO_MEASURE")
)
merged_df

,S,R,MP,FROM_MEASURE,TO_MEASURE,AADT2023,SUTRK2023,CUTRK2023
0,0006_000.0,0006,0.0,0.0,46.0,457,0.249610,0.232449
46,0006_000.7,0006,0.7,0.0,46.0,457,0.249610,0.232449
92,0006_016.0,0006,16.0,0.0,46.0,457,0.249610,0.232449
139,0006_046.0,0006,46.0,46.0,77.6,409,0.175063,0.333819
185,0006_060.2,0006,60.2,46.0,77.6,409,0.175063,0.333819
...,...,...,...,...,...,...,...,...
115520,0491_011.6,0491,11.6,2.0,17.1,3558,0.063942,0.330758
115523,0491_012.6,0491,12.6,2.0,17.1,3558,0.063942,0.330758
115526,0491_014.7,0491,14.7,2.0,17.1,3558,0.063942,0.330758
115529,0491_015.8,0491,15.8,2.0,17.1,3558,0.063942,0.330758


In [5]:
merged_df['S'].drop_duplicates().shape

(3092,)

In [6]:
output_json_path = Path(r"_site-traffic-volume-map\widgets\ForecastSidebar\data\truckdata.json")

(
    merged_df[['S', 'AADT2023', 'SUTRK2023', 'CUTRK2023']]
    .rename(columns={'AADT2023': 'V', 'SUTRK2023': 'SU', 'CUTRK2023': 'CU'})
    .set_index('S')
    .to_json(output_json_path, orient='index', indent=2)
)

print(f"Exported merged data to JSON: {output_json_path.resolve()}")


Exported merged data to JSON: D:\GitHub\FORECAST-Traffic-Volume\_site-traffic-volume-map\widgets\ForecastSidebar\data\truckdata.json
